In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.interpolate import interp1d
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

BASE_PATH = "/kaggle/input/asl-signs"
PREPROCESSED_PATH = "./preprocessed_data"

# Load metadata
train_csv = pd.read_csv(f"{BASE_PATH}/train.csv")
with open(f"{BASE_PATH}/sign_to_prediction_index_map.json", "r") as f:
    label_map = json.load(f)

NUM_CLASSES = len(label_map)
print(f"Classes: {NUM_CLASSES}")

N_JOBS = 4

Classes: 250


In [2]:
def has_valid_hands_fast(sample_path, min_frames=5):
    """Filter samples with insufficient hand data"""
    full_path = os.path.join(BASE_PATH, sample_path)
    df = pd.read_parquet(full_path, columns=["frame", "type"], engine="pyarrow")
    
    hand_df = df[df["type"].isin(["left_hand", "right_hand"])]
    if hand_df.empty or hand_df["frame"].nunique() < min_frames:
        return False
    return True

def check_index(i):
    return i if has_valid_hands_fast(train_csv.iloc[i]["path"]) else None

# Try loading pre-computed indices, otherwise compute
try:
    valid_indices = np.load("/kaggle/input/valid-indices-npy/valid_indices.npy").tolist()
    print(f"Loaded {len(valid_indices)} valid indices from cache")
except:
    print("Computing valid indices (this may take a while)...")
    results = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(check_index)(i) for i in tqdm(range(len(train_csv)))
    )
    valid_indices = [i for i in results if i is not None]
    os.makedirs("/kaggle/working", exist_ok=True)
    np.save("/kaggle/working/valid_indices.npy", np.array(valid_indices))
    print(f"Valid samples: {len(valid_indices)} / {len(train_csv)}")

# Filter CSV to valid samples
train_csv = train_csv.iloc[valid_indices].reset_index(drop=True)
print(f"Filtered train_csv length: {len(train_csv)}")


class LandmarkPreprocessor:
    @staticmethod
    def filter_empty_frames(kp, threshold=0.5):
        missing_per_frame = np.sum(kp == 0, axis=(1, 2)) / (kp.shape[1] * 3)
        valid_mask = missing_per_frame < threshold
        if np.sum(valid_mask) == 0:
            return kp[:1]
        return kp[valid_mask]
    
    @staticmethod
    def interpolate_missing(kp):
        T, L, C = kp.shape
        kp_interp = kp.copy()
        
        for l in range(L):
            for c in range(C):
                track = kp[:, l, c]
                valid_mask = track != 0
                n_valid = np.sum(valid_mask)
                
                if n_valid == 0:
                    continue
                elif n_valid == 1:
                    kp_interp[:, l, c] = track[valid_mask][0]
                else:
                    valid_indices = np.where(valid_mask)[0]
                    valid_values = track[valid_mask]
                    f = interp1d(valid_indices, valid_values, kind='linear',
                               fill_value='extrapolate', bounds_error=False)
                    kp_interp[:, l, c] = f(np.arange(T))
        
        return kp_interp
    
    @staticmethod
    def anchor_normalize(kp):
        T, L, C = kp.shape
        
        # For hand landmarks, use centroid of all valid points
        valid_mask = kp != 0
        
        # Calculate centroid per frame
        anchor_pos = np.zeros((T, C))
        scale = np.zeros(T)
        
        for t in range(T):
            valid_points = kp[t][np.any(valid_mask[t], axis=1)]
            if len(valid_points) > 0:
                anchor_pos[t] = valid_points.mean(axis=0)
                # Scale based on spread of points
                distances = np.linalg.norm(valid_points - anchor_pos[t], axis=1)
                scale[t] = distances.mean() if len(distances) > 0 else 1.0
            else:
                anchor_pos[t] = 0
                scale[t] = 1.0
        
        # Apply normalization
        anchor_pos_expanded = anchor_pos[:, np.newaxis, :]
        kp_relative = kp - anchor_pos_expanded
        
        # Use mean scale to avoid division by zero
        mean_scale = scale[scale > 0].mean() if np.any(scale > 0) else 1.0
        if mean_scale > 1e-6:
            kp_normalized = kp_relative / mean_scale
        else:
            kp_normalized = kp_relative
        
        return kp_normalized, anchor_pos, mean_scale

class LandmarkAugmentor:
    @staticmethod
    def horizontal_flip(kp, left_hand_indices, right_hand_indices, p=0.5):
        """FIXED: Proper handling of hand indices"""
        if np.random.rand() < p:
            kp = kp.copy()
            kp[:, :, 0] *= -1  # Flip x-coordinate
            
            # Swap hands if both are present
            if len(left_hand_indices) > 0 and len(right_hand_indices) > 0:
                left_hand = kp[:, left_hand_indices, :].copy()
                right_hand = kp[:, right_hand_indices, :].copy()
                kp[:, left_hand_indices, :] = right_hand
                kp[:, right_hand_indices, :] = left_hand
        return kp
    
    @staticmethod
    def rotation_3d(kp, max_angle=60, p=0.5):
        if np.random.rand() < p:
            angle = np.random.uniform(-max_angle, max_angle)
            angle_rad = np.deg2rad(angle)
            
            cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
            R = np.array([[cos_a, -sin_a, 0], [sin_a, cos_a, 0], [0, 0, 1]])
            
            T, L, _ = kp.shape
            kp_flat = kp.reshape(-1, 3)
            kp_rotated = (R @ kp_flat.T).T
            kp = kp_rotated.reshape(T, L, 3)
        return kp
    
    @staticmethod
    def resize_3d(kp, scale_range=(0.8, 1.2), p=0.5):
        if np.random.rand() < p:
            scale = np.random.uniform(*scale_range)
            kp = kp * scale
        return kp
    
    @staticmethod
    def finger_dropout(kp, hand_indices, p_per_finger=0.1, p=0.3):
        if np.random.rand() < p:
            for idx in hand_indices:
                if np.random.rand() < p_per_finger:
                    kp[:, idx, :] = 0
        return kp

Computing valid indices (this may take a while)...


100%|██████████| 94477/94477 [08:47<00:00, 179.12it/s]

Valid samples: 94198 / 94477
Filtered train_csv length: 94198


In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=logits.shape[1])
        p_t = (probs * targets_one_hot).sum(dim=1)
        
        focal_weight = (1 - p_t) ** self.gamma
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        focal_loss = focal_weight * ce_loss
        
        if self.alpha is not None:
            if self.alpha.device != logits.device:
                self.alpha = self.alpha.to(logits.device)
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [7]:
class ASLDataset(Dataset):
    def __init__(self, dataframe, label_map, max_frames=64, split='train',
                 use_anchor_norm=True, use_interpolation=True, use_augmentation=True):
        self.df = dataframe.reset_index(drop=True)
        self.label_map = label_map
        self.max_frames = max_frames
        self.split = split
        self.use_anchor_norm = use_anchor_norm
        self.use_interpolation = use_interpolation
        self.use_augmentation = use_augmentation and (split == 'train')
        
        # Initialize helpers
        self.preprocessor = LandmarkPreprocessor()
        self.augmentor = LandmarkAugmentor()
        
        # CRITICAL: Define hand indices (42 landmarks total, 21 per hand)
        self.left_hand_indices = list(range(0, 21))
        self.right_hand_indices = list(range(21, 42))
        
        # Class balancing
        self.class_counts = self._analyze_class_distribution()
        self.class_weights = self._compute_class_weights()
        

        print(f"ASL Dataset - {split.upper()}")
        
        print(f"Samples: {len(self.df)}")
        print(f"Classes: {len(label_map)}")
        print(f"Max frames: {max_frames}")
        print(f"Anchor normalization: {use_anchor_norm}")
        print(f"Interpolation: {use_interpolation}")
        print(f"Augmentation: {self.use_augmentation}")

    
    def _analyze_class_distribution(self):
        labels = [self.label_map[row['sign']] for _, row in self.df.iterrows()]
        return Counter(labels)
    
    def _compute_class_weights(self):
        num_classes = len(self.label_map)
        weights = np.zeros(num_classes)
        total_samples = sum(self.class_counts.values())
        
        for class_idx in range(num_classes):
            count = self.class_counts.get(class_idx, 1)
            weights[class_idx] = total_samples / (num_classes * count)
        
        weights = weights / weights.sum() * num_classes
        return torch.FloatTensor(weights)
    
    def get_sample_weights(self):
        """For WeightedRandomSampler"""
        sample_weights = []
        for _, row in self.df.iterrows():
            label = self.label_map[row['sign']]
            sample_weights.append(self.class_weights[label].item())
        return torch.FloatTensor(sample_weights)
    
    def load_hand_keypoints(self, sample_path):
        """Load hand landmarks (42 landmarks: 21 left + 21 right)"""
        full_path = os.path.join(BASE_PATH, sample_path)
        df = pd.read_parquet(full_path, engine="pyarrow")
        
        hand_df = df[df["type"].isin(["left_hand", "right_hand"])]
        hand_df = hand_df.sort_values(["frame", "type", "landmark_index"])
        
        frames = hand_df["frame"].nunique()
        coords = hand_df[["x", "y", "z"]].values
        coords = coords.reshape(frames, 42, 3)
        
        return np.nan_to_num(coords, nan=0.0)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load keypoints
        kp = self.load_hand_keypoints(row["path"])  # (T, 42, 3)
        original_length = kp.shape[0]
        
        # Preprocessing
        kp = self.preprocessor.filter_empty_frames(kp, threshold=0.5)
        
        if self.use_interpolation:
            kp = self.preprocessor.interpolate_missing(kp)
        
        if self.use_anchor_norm:
            kp, _, _ = self.preprocessor.anchor_normalize(kp)
        
        # Temporal resampling
        if kp.shape[0] < self.max_frames:
            pad_len = self.max_frames - kp.shape[0]
            kp = np.concatenate([kp, np.zeros((pad_len, 42, 3))], axis=0)
        elif kp.shape[0] > self.max_frames:
            indices = np.linspace(0, kp.shape[0]-1, self.max_frames).astype(int)
            kp = kp[indices]
        
        kp = np.nan_to_num(kp, nan=0.0)
        
        # Augmentation (FIXED with correct indices)
        if self.use_augmentation:
            kp = self.augmentor.horizontal_flip(
                kp, self.left_hand_indices, self.right_hand_indices, p=0.5)
            kp = self.augmentor.rotation_3d(kp, max_angle=60, p=0.5)
            kp = self.augmentor.resize_3d(kp, scale_range=(0.8, 1.2), p=0.5)
            
            all_hand_indices = self.left_hand_indices + self.right_hand_indices
            kp = self.augmentor.finger_dropout(kp, all_hand_indices, p=0.3)
        
        label = self.label_map[row["sign"]]
        
        return torch.FloatTensor(kp), label


In [10]:
def create_dataloaders(train_df, val_df, label_map, batch_size=32, 
                       max_frames=64, num_workers=4):
    
    train_ds = ASLDataset(
        train_df, label_map, max_frames=max_frames, split='train',
        use_anchor_norm=True, use_interpolation=True, use_augmentation=True
    )
    
    val_ds = ASLDataset(
        val_df, label_map, max_frames=max_frames, split='val',
        use_anchor_norm=True, use_interpolation=True, use_augmentation=False
    )
    
    # Weighted sampler for balanced training
    sample_weights = train_ds.get_sample_weights()
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=sampler,
        num_workers=num_workers, pin_memory=True,
        prefetch_factor=2, persistent_workers=True, drop_last=True
    )
    
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        prefetch_factor=2, persistent_workers=True
    )
    
    
    focal_loss = FocalLoss(alpha=train_ds.class_weights, gamma=2.0)
    
    return train_loader, val_loader, focal_loss, train_ds.class_weights


In [11]:
if __name__ == "__main__":
    set_seed(42)

    from sklearn.model_selection import train_test_split

    # 🔥 CHANGE 1: REMOVE manual 80/20 slicing (this was NOT stratified)
    # split_idx = int(len(train_csv) * 0.8)
    # train_df = train_csv[:split_idx]
    # val_df = train_csv[split_idx:]

    # 🔥 CHANGE 2: Use STRATIFIED split so every class appears in both sets
    train_df, val_df = train_test_split(
        train_csv,
        test_size=0.25,                 # 75% train / 25% val
        stratify=train_csv["sign"],     # keeps class balance
        random_state=42
    )

    print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}")

    train_loader, val_loader, focal_loss, class_weights = create_dataloaders(
        train_df, val_df, label_map,
        batch_size=32, max_frames=64, num_workers=4
    )

    # Quick sanity check
    for batch_idx, (data, labels) in enumerate(train_loader):
        print(f"Batch shape: {data.shape}")  # Expected: [32, 64, 42, 3]
        print(f"Labels shape: {labels.shape}")
        break


Train samples: 70648, Val samples: 23550
ASL Dataset - TRAIN
Samples: 70648
Classes: 250
Max frames: 64
Anchor normalization: True
Interpolation: True
Augmentation: True
ASL Dataset - VAL
Samples: 23550
Classes: 250
Max frames: 64
Anchor normalization: True
Interpolation: True
Augmentation: False
Batch shape: torch.Size([32, 64, 42, 3])
Labels shape: torch.Size([32])


In [18]:
import math

def stratified_sample_dataset(train_csv, sample_ratio=0.75, random_state=42):
    """
    Sample dataset while preserving class distribution
    """
    print(f"Stratified Sampling: Keeping {sample_ratio*100:.0f}% of data")
    
    sampled_dfs = []
    
    for sign_class in train_csv['sign'].unique():
        class_df = train_csv[train_csv['sign'] == sign_class]
        n_samples = max(1, int(len(class_df) * sample_ratio))
        
        sampled_class = class_df.sample(n=n_samples, random_state=random_state)
        sampled_dfs.append(sampled_class)
    
    sampled_df = pd.concat(sampled_dfs, ignore_index=True)
    sampled_df = sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    print(f"Original dataset: {len(train_csv)} samples")
    print(f"Sampled dataset: {len(sampled_df)} samples")
    print(f"Reduction: {(1 - len(sampled_df)/len(train_csv))*100:.1f}%")
    
    orig_dist = train_csv['sign'].value_counts(normalize=True).sort_index()
    samp_dist = sampled_df['sign'].value_counts(normalize=True).sort_index()
    
    max_deviation = (orig_dist - samp_dist).abs().max()
    print(f"Max class distribution deviation: {max_deviation*100:.2f}%")
    print(f"{'='*60}\n")
    
    return sampled_df


In [19]:
class DifficultyAnalyzer:
    """
    Multi-dimensional difficulty scoring for intelligent cascading
    Combines: prediction uncertainty, spatial complexity, temporal complexity, motion patterns
    """
    
    @staticmethod
    def compute_prediction_uncertainty(logits):
        """Entropy-based uncertainty from model predictions"""
        probs = F.softmax(logits, dim=1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1)
        max_entropy = math.log(logits.shape[1])
        return entropy / max_entropy  # [0, 1]
    
    @staticmethod
    def compute_spatial_complexity(x):
        """
        Measure spatial spread and hand coordination complexity
        Args: x shape (batch, max_len, 42, 3)
        Returns: complexity score [0, 1]
        """
        batch_size = x.shape[0]
        complexity_scores = []
        
        for i in range(batch_size):
            sample = x[i]  # (max_len, 42, 3)
            
            # Get valid frames (non-zero)
            valid_mask = sample.abs().sum(dim=(1, 2)) > 0
            if valid_mask.sum() == 0:
                complexity_scores.append(0.0)
                continue
            
            valid_frames = sample[valid_mask]  # (T, 42, 3)
            
            # 1. Hand spread (variance of landmark positions)
            spread = valid_frames.std(dim=1).mean().item()  # Higher = more spread out
            
            # 2. Two-hand coordination (difference between left and right hand)
            left_hand = valid_frames[:, :21, :]  # First 21 landmarks
            right_hand = valid_frames[:, 21:, :]  # Last 21 landmarks
            
            left_center = left_hand.mean(dim=1)
            right_center = right_hand.mean(dim=1)
            hand_distance = torch.norm(left_center - right_center, dim=1).mean().item()
            
            # 3. Finger articulation (variance within each hand)
            left_articulation = left_hand.std(dim=1).mean().item()
            right_articulation = right_hand.std(dim=1).mean().item()
            articulation = (left_articulation + right_articulation) / 2
            
            # Normalize and combine (higher values = more complex)
            spatial_complexity = (spread * 0.3 + hand_distance * 0.4 + articulation * 0.3)
            # Clip to [0, 1] range (empirically tuned)
            spatial_complexity = min(spatial_complexity / 2.0, 1.0)
            
            complexity_scores.append(spatial_complexity)
        
        return torch.tensor(complexity_scores, device=x.device)
    
    @staticmethod
    def compute_temporal_complexity(x):
        """
        Measure temporal variation and motion smoothness
        Args: x shape (batch, max_len, 42, 3)
        Returns: complexity score [0, 1]
        """
        batch_size = x.shape[0]
        complexity_scores = []
        
        for i in range(batch_size):
            sample = x[i]  # (max_len, 42, 3)
            
            # Get valid frames
            valid_mask = sample.abs().sum(dim=(1, 2)) > 0
            if valid_mask.sum() <= 1:
                complexity_scores.append(0.0)
                continue
            
            valid_frames = sample[valid_mask]  # (T, 42, 3)
            T = valid_frames.shape[0]
            
            # 1. Frame-to-frame velocity (first derivative)
            velocity = torch.diff(valid_frames, dim=0)  # (T-1, 42, 3)
            avg_velocity = velocity.abs().mean().item()
            
            # 2. Acceleration (second derivative) - jerkiness
            if T > 2:
                acceleration = torch.diff(velocity, dim=0)  # (T-2, 42, 3)
                avg_acceleration = acceleration.abs().mean().item()
            else:
                avg_acceleration = 0.0
            
            # 3. Temporal variance (how much the sign changes over time)
            temporal_variance = valid_frames.std(dim=0).mean().item()
            
            # 4. Direction changes (non-smooth motion)
            if T > 2:
                velocity_norm = F.normalize(velocity.reshape(T-1, -1), dim=1)
                direction_changes = (1 - (velocity_norm[:-1] * velocity_norm[1:]).sum(dim=1)).mean().item()
            else:
                direction_changes = 0.0
            
            # Combine metrics (higher = more complex temporal pattern)
            temporal_complexity = (
                avg_velocity * 0.3 + 
                avg_acceleration * 0.3 + 
                temporal_variance * 0.2 + 
                direction_changes * 0.2
            )
            # Normalize to [0, 1]
            temporal_complexity = min(temporal_complexity / 1.5, 1.0)
            
            complexity_scores.append(temporal_complexity)
        
        return torch.tensor(complexity_scores, device=x.device)
    
    @staticmethod
    def compute_motion_pattern_complexity(x):
        """
        Analyze motion patterns (circular, linear, static, etc.)
        Args: x shape (batch, max_len, 42, 3)
        Returns: complexity score [0, 1]
        """
        batch_size = x.shape[0]
        complexity_scores = []
        
        for i in range(batch_size):
            sample = x[i]
            
            valid_mask = sample.abs().sum(dim=(1, 2)) > 0
            if valid_mask.sum() <= 2:
                complexity_scores.append(0.0)
                continue
            
            valid_frames = sample[valid_mask]
            T = valid_frames.shape[0]
            
            # Use hand centroids for motion pattern
            hand_centroid = valid_frames.mean(dim=1)  # (T, 3)
            
            # 1. Path length vs straight-line distance (tortuosity)
            path_length = torch.norm(torch.diff(hand_centroid, dim=0), dim=1).sum().item()
            straight_distance = torch.norm(hand_centroid[-1] - hand_centroid[0]).item()
            
            if straight_distance > 1e-6:
                tortuosity = path_length / straight_distance
            else:
                tortuosity = 1.0
            
            # 2. 3D motion (z-axis usage)
            z_variance = hand_centroid[:, 2].std().item()
            
            # 3. Circular vs linear motion (variance in different axes)
            xy_variance = hand_centroid[:, :2].std(dim=0).mean().item()
            
            # Combine (complex motions have high tortuosity and 3D usage)
            motion_complexity = (
                min(tortuosity / 3.0, 1.0) * 0.5 +
                min(z_variance / 0.3, 1.0) * 0.3 +
                min(xy_variance / 0.5, 1.0) * 0.2
            )
            
            complexity_scores.append(motion_complexity)
        
        return torch.tensor(complexity_scores, device=x.device)
    
    @classmethod
    def compute_difficulty_score(cls, x, logits, weights=None):
        """
        Compute comprehensive difficulty score combining all factors
        
        Args:
            x: input data (batch, max_len, 42, 3)
            logits: model predictions (batch, num_classes)
            weights: dict of weights for each component
        
        Returns:
            difficulty_score: [0, 1] where higher = more difficult
            components: dict with individual scores for analysis
        """
        if weights is None:
            weights = {
                'prediction_uncertainty': 0.35,
                'spatial_complexity': 0.25,
                'temporal_complexity': 0.25,
                'motion_complexity': 0.15
            }
        
        # Compute all components
        pred_uncertainty = cls.compute_prediction_uncertainty(logits)
        spatial_comp = cls.compute_spatial_complexity(x)
        temporal_comp = cls.compute_temporal_complexity(x)
        motion_comp = cls.compute_motion_pattern_complexity(x)
        
        # Weighted combination
        difficulty = (
            weights['prediction_uncertainty'] * pred_uncertainty +
            weights['spatial_complexity'] * spatial_comp +
            weights['temporal_complexity'] * temporal_comp +
            weights['motion_complexity'] * motion_comp
        )
        
        components = {
            'prediction_uncertainty': pred_uncertainty,
            'spatial_complexity': spatial_comp,
            'temporal_complexity': temporal_comp,
            'motion_complexity': motion_comp,
            'overall_difficulty': difficulty
        }
        
        return difficulty, components


In [20]:

class Conv1DBlock(nn.Module):
    """1D Convolutional block with depthwise separable convolution"""
    def __init__(self, dim, kernel_size=17, drop_rate=0.2):
        super().__init__()
        padding = kernel_size // 2
        
        self.dwconv = nn.Conv1d(dim, dim, kernel_size, padding=padding, groups=dim)
        self.pwconv = nn.Conv1d(dim, dim, 1)
        self.bn = nn.BatchNorm1d(dim, momentum=0.05)
        self.dropout = nn.Dropout(drop_rate)
        
    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = self.pwconv(x)
        x = self.bn(x)
        x = F.gelu(x)
        x = self.dropout(x)
        return x + residual


class LightweightTransformerBlock(nn.Module):
    """Lightweight transformer block for temporal modeling"""
    def __init__(self, dim, num_heads=4, expand=2, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * expand),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * expand, dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x.transpose(1, 2)
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x.transpose(1, 2)


class LateDropout(nn.Module):
    """Dropout that only activates after a certain training step"""
    def __init__(self, p=0.8, start_step=0):
        super().__init__()
        self.p = p
        self.start_step = start_step
        self.current_step = 0
    
    def forward(self, x):
        if self.training and self.current_step >= self.start_step:
            return F.dropout(x, p=self.p, training=True)
        return x
    
    def step(self):
        self.current_step += 1

In [23]:
class ASLFilterModel(nn.Module):
    """
    Lightweight first-pass filter with multi-dimensional difficulty assessment
    """
    def __init__(self, num_classes, channels=126, dim=96, dropout_step=0):
        super().__init__()
        self.channels = channels

        # Stem
        self.stem_conv = nn.Linear(channels, dim, bias=False)
        self.stem_bn = nn.BatchNorm1d(dim, momentum=0.1)

        # Block 1
        self.conv_block1 = nn.Sequential(
            Conv1DBlock(dim, kernel_size=17, drop_rate=0.2),
            Conv1DBlock(dim, kernel_size=17, drop_rate=0.2),
            Conv1DBlock(dim, kernel_size=17, drop_rate=0.2),
        )
        self.transformer1 = LightweightTransformerBlock(dim, num_heads=4, expand=2)

        # Block 2
        self.conv_block2 = nn.Sequential(
            Conv1DBlock(dim, kernel_size=17, drop_rate=0.2),
            Conv1DBlock(dim, kernel_size=17, drop_rate=0.2),
            Conv1DBlock(dim, kernel_size=17, drop_rate=0.2),
        )
        self.transformer2 = LightweightTransformerBlock(dim, num_heads=4, expand=2)

        # Head
        self.top_conv = nn.Linear(dim, dim * 2)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.late_dropout = LateDropout(0.8, start_step=dropout_step)
        self.classifier = nn.Linear(dim * 2, num_classes)

        self.difficulty_analyzer = DifficultyAnalyzer()
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x, return_difficulty=False, difficulty_weights=None):
        batch_size = x.size(0)
        x_original = x.detach()   # FIX: no clone

        x = x.view(batch_size, x.size(1), -1)

        x = self.stem_conv(x)
        x = self.stem_bn(x.transpose(1, 2))

        x = self.transformer1(self.conv_block1(x))
        x = self.transformer2(self.conv_block2(x))

        x = F.gelu(self.top_conv(x.transpose(1, 2)))
        x = self.global_pool(x.transpose(1, 2)).squeeze(-1)

        x = self.late_dropout(x)
        logits = self.classifier(x)

        if return_difficulty:
            difficulty, components = self.difficulty_analyzer.compute_difficulty_score(
                x_original, logits, difficulty_weights
            )
            return logits, difficulty, components

        return logits

    def get_model_size_mb(self):
        return (sum(p.numel() * p.element_size() for p in self.parameters()) +
                sum(b.numel() * b.element_size() for b in self.buffers())) / (1024 ** 2)


In [25]:
def train_filter_model(
    model, train_loader, val_loader, criterion,
    num_epochs=100, lr=1e-3, device='cuda',
    difficulty_threshold=0.4,
    warmup_epochs=10,
    difficulty_start_epoch=20,
    difficulty_weights=None,
    label_smoothing=0.1,
    save_dir="/kaggle/working"
):
    model.to(device)
    ce_criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr * 0.01
    )

    best_val_acc = 0.0
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(num_epochs):

        # 🔴 FIX: LateDropout stepped ONCE per epoch
        if hasattr(model, 'late_dropout'):
            model.late_dropout.step()

        model.train()
        train_correct, train_total, train_loss = 0, 0, 0

        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()

            logits = model(data)
            loss = ce_criterion(logits, labels) if epoch < warmup_epochs else criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            train_correct += (logits.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_acc = train_correct / train_total

        model.eval()
        val_correct, val_total, val_loss = 0, 0, 0

        with torch.no_grad():
            for data, labels in val_loader:
                data, labels = data.to(device), labels.to(device)
                logits = model(data)
                loss = ce_criterion(logits, labels)

                val_loss += loss.item()
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        scheduler.step()

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Acc: {train_acc*100:.2f}% | "
              f"Val Acc: {val_acc*100:.2f}%")

        # ✅ Save BEST
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{save_dir}/best_filter_model.pth")

        # ✅ Save EVERY 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save(
                model.state_dict(),
                f"{save_dir}/filter_epoch_{epoch+1}.pth"
            )

    print(f"Training done. Best Val Acc: {best_val_acc*100:.2f}%")
